In [2]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find the repository root.
ROOT = Path.cwd()
while not (ROOT / "data_labeling").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Repository root was not found.")
    ROOT = ROOT.parent

INPUT_FILE = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "labels_noaa_2010_2026_binary.csv"
)

# Keep the new NOAA Stage 1 experiment separate from older experiments.
OUTPUT_DIR = (
    ROOT
    / "data_labeling"
    / "data_labels"
    / "noaa"
    / "stage1_experiment"
)

assert INPUT_FILE.exists(), f"File not found: {INPUT_FILE}"

print("Input:", INPUT_FILE)
print("Output:", OUTPUT_DIR)

Input: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/labels_noaa_2010_2026_binary.csv
Output: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/stage1_experiment


In [3]:
labels = pd.read_csv(INPUT_FILE)

# Extract image observation times from the paths.
timestamp_text = labels["label"].str.extract(
    r"HMI\.m(\d{4}\.\d{2}\.\d{2}_\d{2}\.\d{2}\.\d{2})"
)[0]

labels["timestamp"] = pd.to_datetime(
    timestamp_text,
    format="%Y.%m.%d_%H.%M.%S",
    errors="coerce",
)

labels["goes_class"] = pd.to_numeric(
    labels["goes_class"],
    errors="coerce",
)

# Validate the complete source file.
assert labels["timestamp"].notna().all()
assert labels["goes_class"].isin([0, 1]).all()
assert not labels["label"].duplicated().any()

labels = labels.sort_values("timestamp").reset_index(drop=True)

print("Images:", f"{len(labels):,}")
print("First:", labels["timestamp"].min())
print("Last:", labels["timestamp"].max())

Images: 128,117
First: 2010-12-06 07:00:00
Last: 2026-07-31 23:00:00


In [4]:
# Selected Stage 1 training period.
STAGE1_TRAIN = (
    pd.Timestamp("2010-12-06 07:00:00"),
    pd.Timestamp("2017-01-06 03:00:00"),
)

# Complete 24-hour NF-only gap.
STAGE1_GAP = (
    pd.Timestamp("2017-01-06 04:00:00"),
    pd.Timestamp("2017-01-07 03:00:00"),
)

# Natural chronological Stage 1 holdout.
STAGE1_HOLDOUT = (
    pd.Timestamp("2017-01-07 04:00:00"),
    pd.Timestamp("2017-12-31 23:00:00"),
)

# Common future validation period for comparing trials.
VALIDATION = (
    pd.Timestamp("2024-07-01 00:00:00"),
    pd.Timestamp("2025-06-30 23:00:00"),
)

# Complete separation between validation and test.
VALIDATION_TEST_GAP = (
    pd.Timestamp("2025-07-01 00:00:00"),
    pd.Timestamp("2025-07-01 23:00:00"),
)

# Final future test period.
TEST = (
    pd.Timestamp("2025-07-02 00:00:00"),
    pd.Timestamp("2026-07-31 23:00:00"),
)

In [5]:
def select_period(start, end):
    """Select an inclusive chronological period."""
    return labels[
        labels["timestamp"].between(start, end)
    ].copy()


stage1_train = select_period(*STAGE1_TRAIN)
stage1_gap = select_period(*STAGE1_GAP)
stage1_holdout = select_period(*STAGE1_HOLDOUT)

validation = select_period(*VALIDATION)
validation_test_gap = select_period(*VALIDATION_TEST_GAP)
test = select_period(*TEST)

print("Partitions selected.")

Partitions selected.


In [6]:
def summarize(name, frame):
    """Calculate class counts and percentages."""
    total = len(frame)
    fl = int(frame["goes_class"].sum())
    nf = total - fl

    return {
        "dataset": name,
        "start": frame["timestamp"].min(),
        "end": frame["timestamp"].max(),
        "images": total,
        "NF": nf,
        "FL": fl,
        "NF_percent": 100 * nf / total,
        "FL_percent": 100 * fl / total,
    }


summary = pd.DataFrame([
    summarize("Stage 1 train", stage1_train),
    summarize("Stage 1 gap", stage1_gap),
    summarize("Stage 1 holdout", stage1_holdout),
    summarize("Validation", validation),
    summarize("Validation-test gap", validation_test_gap),
    summarize("Test", test),
])

display(summary)

,dataset,start,end,images,NF,FL,NF_percent,FL_percent
0,Stage 1 train,2010-12-06 07:00:00,2017-01-06 03:00:00,46638,38015,8623,81.510785,18.489215
1,Stage 1 gap,2017-01-06 04:00:00,2017-01-07 03:00:00,24,24,0,100.000000,0.000000
2,Stage 1 holdout,2017-01-07 04:00:00,2017-12-31 23:00:00,8230,7900,330,95.990279,4.009721
3,Validation,2024-07-01 00:00:00,2025-06-30 23:00:00,8480,3299,5181,38.903302,61.096698
4,Validation-test gap,2025-07-01 00:00:00,2025-07-01 23:00:00,24,24,0,100.000000,0.000000
5,Test,2025-07-02 00:00:00,2026-07-31 23:00:00,9187,6019,3168,65.516491,34.483509


In [8]:
# Validate both 24-hour gaps
def validate_gap(name, frame, start, end):
    """Require 24 complete hourly NF observations."""
    expected_hours = pd.date_range(start, end, freq="h")
    missing_hours = expected_hours.difference(frame["timestamp"])

    assert len(frame) == 24, f"{name}: expected 24 images"
    assert frame["goes_class"].sum() == 0, f"{name}: contains FL labels"
    assert len(missing_hours) == 0, f"{name}: contains missing hours"

    print(f"{name}: valid 24-hour NF-only gap")


validate_gap(
    "Stage 1 train-holdout gap",
    stage1_gap,
    *STAGE1_GAP,
)

validate_gap(
    "Validation-test gap",
    validation_test_gap,
    *VALIDATION_TEST_GAP,
)

Stage 1 train-holdout gap: valid 24-hour NF-only gap
Validation-test gap: valid 24-hour NF-only gap


In [9]:
# Check chronological ordering and overlap
partitions = {
    "Stage1_train": stage1_train,
    "Stage1_holdout": stage1_holdout,
    "Validation": validation,
    "Test": test,
}

# Confirm every partition contains both classes.
for name, frame in partitions.items():
    assert set(frame["goes_class"].unique()) == {0, 1}, (
        f"{name} does not contain both classes"
    )

# Confirm no image path appears in multiple partitions.
names = list(partitions)

for index, first_name in enumerate(names):
    first_paths = set(partitions[first_name]["label"])

    for second_name in names[index + 1:]:
        second_paths = set(partitions[second_name]["label"])
        overlap = first_paths & second_paths

        assert not overlap, (
            f"{first_name} and {second_name} overlap"
        )

# Confirm chronological ordering.
assert stage1_train["timestamp"].max() < stage1_holdout["timestamp"].min()
assert validation["timestamp"].max() < test["timestamp"].min()

print("Both classes are present in every partition.")
print("No image paths overlap.")
print("Chronological ordering is valid.")

Both classes are present in every partition.
No image paths overlap.
Chronological ordering is valid.


In [10]:
output_frames = {
    "Stage1_train.csv": stage1_train[["label", "goes_class"]],
    "Stage1_holdout.csv": stage1_holdout[["label", "goes_class"]],
    "validation_2024_2025.csv": validation[["label", "goes_class"]],
    "test_2025_2026_july.csv": test[["label", "goes_class"]],
}

for filename, frame in output_frames.items():
    print(filename, frame.shape)

Stage1_train.csv (46638, 2)
Stage1_holdout.csv (8230, 2)
validation_2024_2025.csv (8480, 2)
test_2025_2026_july.csv (9187, 2)


In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for filename, frame in output_frames.items():
    output_path = OUTPUT_DIR / filename
    frame.to_csv(output_path, index=False)
    print("Created:", output_path)

Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/stage1_experiment/Stage1_train.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/stage1_experiment/Stage1_holdout.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/stage1_experiment/validation_2024_2025.csv
Created: /Users/piyushluitel/Desktop/PhD/Research/Full-Disk-Attention---Continual-Learning-/data_labeling/data_labels/noaa/stage1_experiment/test_2025_2026_july.csv
